# PV-labelled customers and negative net load

Run cells in order. Each cell reports records and categories. The scan is opt-in because it streams approximately 77 GiB.

In [ ]:
from pathlib import Path
import json, re
import numpy as np
import pandas as pd
pd.set_option('display.max_colwidth', 100)
def data_root():
    for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (p/'store'/'input_data').exists(): return p/'store'/'input_data'
    raise FileNotFoundError('Start from the repository or a child directory.')
DATA_ROOT=data_root()
GIGI=DATA_ROOT/'HackDays2026 - GIGI.csv'
ZGP=DATA_ROOT/'Zähler-GP.csv'
MAPPING=DATA_ROOT/'mpid_zähler_mapping.csv'
print('Data root:', DATA_ROOT)
def text(s): return s.astype('string').str.strip()
def present(v): return pd.notna(v) and str(v).strip().casefold()=='x'


In [ ]:
# 1. Load references, normalize PV labels, and consolidate to one customer row.
g=pd.read_csv(GIGI,sep=';',encoding='utf-8-sig',dtype='string'); z=pd.read_csv(ZGP,sep=';',encoding='utf-8-sig',dtype='string'); m=pd.read_csv(MAPPING,sep=';',encoding='utf-8-sig',dtype='string')
print(f'GIGI: {len(g):,}; Zähler-GP: {len(z):,}; mapping: {len(m):,}')
print('GIGI columns:', list(g.columns))
g['_row']=range(len(g)); g['gp_nr']=text(g['GP-Nr']); g=g[g.gp_nr.notna() & g.gp_nr.ne('')].copy()
assets={'heat_pump':'WärmePumpe','pv':' PV','battery_storage':'Batterie/Speicher','ev_charger':'Ladestation für Elektrofahrzeuge','heat_pump_boiler':'Wärmepumpenboiler'}
assets={k:v for k,v in assets.items() if v in g.columns}; g['pv_row']=g[assets['pv']].map(present)
g['inbetrieb_date']=pd.to_datetime(g.get('InBetrieb-Datum'),errors='coerce',dayfirst=True); g['uebergabe_date']=pd.to_datetime(g.get('Übergabe'),errors='coerce',dayfirst=True)
g['event_date']=g.inbetrieb_date.fillna(g.uebergabe_date); g['date_source']=np.where(g.inbetrieb_date.notna(),'InBetrieb-Datum',np.where(g.uebergabe_date.notna(),'Übergabe','unknown'))
def timeline(rows):
    seen=set(); events={}
    for _,r in rows.sort_values(['event_date','_row'],na_position='last').iterrows():
        now={a for a,c in assets.items() if present(r[c])}; added=sorted(now-seen); seen|=now
        if added:
            date=None if pd.isna(r.event_date) else r.event_date.date().isoformat(); key=(date,r.date_source)
            e=events.setdefault(key,{'date':date,'assets_added':[],'date_source':r.date_source})
            e['assets_added'] += [a for a in added if a not in e['assets_added']]
    return list(events.values())
tl=g.groupby('gp_nr',sort=False).apply(timeline,include_groups=False).rename('asset_additions').reset_index()
customers=g.groupby('gp_nr',as_index=False).agg(gigi_record_count=('_row','size'),pv_positive=('pv_row','any'),inbetrieb_dates=('inbetrieb_date',lambda x:sorted({d.date().isoformat() for d in x.dropna()})),uebergabe_dates=('uebergabe_date',lambda x:sorted({d.date().isoformat() for d in x.dropna()}))).merge(tl,on='gp_nr')
customers['asset_additions']=customers.asset_additions.map(json.dumps); pv=customers[customers.pv_positive].copy()
print(f'Customers: {len(customers):,}; PV-positive (any x/X): {len(pv):,}')
display(g[assets['pv']].fillna('<blank>').value_counts().rename_axis('raw PV').to_frame('rows'))


In [ ]:
# 2. Link every PV-positive customer to every meter point; retain ambiguity instead of choosing one.
z=z.rename(columns={'GPartner':'gp_nr','Zählpunktbezeichnung':'designation','Anlage':'anlage'}); m=m.rename(columns={'MP ID':'mp_id','Zählpunktbezeichnung':'designation'})
for f,c in ((z,'gp_nr'),(z,'designation'),(z,'anlage'),(m,'mp_id'),(m,'designation')): f[c]=text(f[c])
designation_rows=z.groupby('designation').size(); links=pv[['gp_nr']].merge(z[['gp_nr','designation','anlage']],on='gp_nr',how='left').merge(m[['designation','mp_id']],on='designation',how='left')
links=links[links.mp_id.notna() & links.mp_id.ne('')].drop_duplicates(['gp_nr','mp_id','designation','anlage']).copy()
links['designation_row_count']=links.designation.map(designation_rows); links['mp_customer_count']=links.mp_id.map(links.groupby('mp_id').gp_nr.nunique())
links['mapping_ambiguous']=links.designation_row_count.gt(1)|links.mp_customer_count.gt(1)
ls=links.groupby('gp_nr',as_index=False).agg(meter_point_count=('mp_id','nunique'),mapping_ambiguity_count=('mapping_ambiguous','sum'),mapped_anlagen=('anlage',lambda x:sorted(set(x.dropna()))))
customer_table=pv.merge(ls,on='gp_nr',how='left'); customer_table[['meter_point_count','mapping_ambiguity_count']]=customer_table[['meter_point_count','mapping_ambiguity_count']].fillna(0).astype(int); customer_table['mapping_ambiguity']=customer_table.mapping_ambiguity_count.gt(0)
print('PV customers with meter points:',(customer_table.meter_point_count>0).sum()); print('Unlinked:',(customer_table.meter_point_count==0).sum()); print('Ambiguous:',customer_table.mapping_ambiguity.sum())
display(customer_table[['gp_nr','meter_point_count','mapping_ambiguity','inbetrieb_dates','uebergabe_dates','asset_additions']].head(10))


In [ ]:
# 3. Discover physical monthly exports (ignore symlinked convenience copies) and profile OBIS codes.
monthly_files=sorted(p for p in DATA_ROOT.rglob('LG_AIM2Hackerdays_kWh_*.csv') if p.is_file() and not p.is_symlink())
print(f'Physical monthly exports: {len(monthly_files)} (documented expectation: 43)')
if len(monthly_files)!=43: print('WARNING: inspect coverage before scanning.')
profiles=[]
for p in monthly_files:
    h=pd.read_csv(p,sep=';',encoding='utf-8-sig',nrows=0)
    if 'OBIS-Code' not in h.columns:
        print('No OBIS-Code; will be skipped safely:',p.name); continue
    s=pd.read_csv(p,sep=';',encoding='utf-8-sig',usecols=['OBIS-Code'],nrows=200_000,dtype='string')
    profiles.append(s['OBIS-Code'].value_counts().rename(p.name)); break
if profiles: display(pd.concat(profiles,axis=1).fillna(0).astype(int))
print('Verify register semantics and units before filling the next cell.')


In [ ]:
# 4. Configure only verified registers. Import/export must have compatible interval units.
NET_LOAD_MODE='separate'  # 'separate' means import - export; alternatively use verified 'signed'
IMPORT_OBIS_CODES=set()
EXPORT_OBIS_CODES=set()
SIGNED_NET_OBIS_CODES=set()
RUN_MONTHLY_SCAN=False
CHUNK_ROWS=100_000
configured=(NET_LOAD_MODE=='separate' and bool(IMPORT_OBIS_CODES) and bool(EXPORT_OBIS_CODES)) or (NET_LOAD_MODE=='signed' and bool(SIGNED_NET_OBIS_CODES))
print('Register configuration ready:',configured); print('Full scan enabled:',RUN_MONTHLY_SCAN)


In [ ]:
# 5. Stream one file/chunk at a time, retaining only linked PV meter points.
def empty_summary(): return pd.DataFrame(columns=['mp_id','valid_interval_count','negative_interval_count','first_negative_timestamp','first_negative_value'])
def scan_file(path,ids):
    h=pd.read_csv(path,sep=';',encoding='utf-8-sig',nrows=0)
    if 'OBIS-Code' not in h: print('Skipped schema without OBIS-Code:',path.name); return empty_summary()
    times=[c for c in h if re.fullmatch(r'\d{2}:\d{2}',str(c))]; codes=(IMPORT_OBIS_CODES|EXPORT_OBIS_CODES) if NET_LOAD_MODE=='separate' else SIGNED_NET_OBIS_CODES; kept=[]
    for c in pd.read_csv(path,sep=';',encoding='utf-8-sig',usecols=['MP ID','OBIS-Code','Datum',*times],chunksize=CHUNK_ROWS,dtype='string'):
        c=c[c['MP ID'].astype('string').str.strip().isin(ids) & c['OBIS-Code'].isin(codes)]
        if not c.empty: kept.append(c)
    if not kept: return empty_summary()
    x=pd.concat(kept).melt(id_vars=['MP ID','Datum','OBIS-Code'],value_vars=times,var_name='interval',value_name='value'); x.value=pd.to_numeric(x.value.astype('string').str.replace(',','.',regex=False),errors='coerce'); key=['MP ID','Datum','interval']
    if NET_LOAD_MODE=='separate':
        imp=x[x['OBIS-Code'].isin(IMPORT_OBIS_CODES)].groupby(key,as_index=False).value.sum(min_count=1).rename(columns={'value':'import'})
        exp=x[x['OBIS-Code'].isin(EXPORT_OBIS_CODES)].groupby(key,as_index=False).value.sum(min_count=1).rename(columns={'value':'export'})
        n=imp.merge(exp,on=key,how='inner'); n['net_load']=n['import']-n['export']
    else: n=x.groupby(key,as_index=False).value.sum(min_count=1).rename(columns={'value':'net_load'}).dropna(subset=['net_load'])
    n['negative']=n.net_load.lt(0); first=n[n.negative].sort_values(key).groupby('MP ID',as_index=False).first()[['MP ID','Datum','interval','net_load']]
    out=n.groupby('MP ID',as_index=False).agg(valid_interval_count=('net_load','size'),negative_interval_count=('negative','sum')).merge(first,on='MP ID',how='left').rename(columns={'MP ID':'mp_id','net_load':'first_negative_value'}); out['first_negative_timestamp']=out['Datum'].astype('string')+' '+out['interval'].astype('string'); return out.drop(columns=['Datum','interval'])
scan_summary=pd.DataFrame()
if not RUN_MONTHLY_SCAN: print('Scan skipped. Set verified codes and RUN_MONTHLY_SCAN=True.')
elif not configured: raise ValueError('Configure verified OBIS code sets before scanning.')
else:
    ids=set(links.mp_id.astype(str)); parts=[]
    for i,p in enumerate(monthly_files,1):
        q=scan_file(p,ids); parts.append(q); print(f'[{i}/{len(monthly_files)}] {p.name}: {len(q):,} meter points')
    scan_summary=pd.concat(parts,ignore_index=True); print('Monthly summaries:',len(scan_summary))


In [ ]:
# 6. One output row per PV-positive customer and visible category counts.
if scan_summary.empty: print('No scan evidence yet; the joined customer table is already available as customer_table.')
else:
    e=scan_summary.groupby('mp_id',as_index=False).agg(valid_interval_count=('valid_interval_count','sum'),negative_interval_count=('negative_interval_count','sum'),first_negative_timestamp=('first_negative_timestamp','min'),first_negative_value=('first_negative_value','min'))
    ce=links[['gp_nr','mp_id']].drop_duplicates().merge(e,on='mp_id',how='left').groupby('gp_nr',as_index=False).agg(valid_interval_count=('valid_interval_count','sum'),negative_interval_count=('negative_interval_count','sum'),first_negative_timestamp=('first_negative_timestamp','min'),first_negative_value=('first_negative_value','min'))
    result=customer_table.merge(ce,on='gp_nr',how='left'); result[['valid_interval_count','negative_interval_count']]=result[['valid_interval_count','negative_interval_count']].fillna(0).astype(int)
    result['negative_load_status']=np.select([result.meter_point_count.eq(0),result.valid_interval_count.eq(0),result.negative_interval_count.gt(0)],['unlinked','linked_but_unobserved','observed_with_negative'],default='observed_without_negative')
    display(result.negative_load_status.value_counts().rename_axis('category').to_frame('customers')); display(result.head(20))


## Limits

This phase retains asset dates and additions but does not filter readings by them. A PV customer can lack negative net load because concurrent consumption exceeds generation; missing or unpaired readings and ambiguous mappings also limit the result.